# 05 — Before / After Analysis

Compares the original trained YOLOv3 against the Tucker-2-compressed
version: parameter count, model size on disk, inference latency, and
detection accuracy (mAP@0.5) on the held-out validation subset.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath("../src"))
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from model import YOLOv3, count_conv_params
from dataset import CocoSubsetDataset, yolo_collate_fn
from tucker_decompose import get_compressible_conv_layers, tucker2_factorize_conv, set_module_by_name
from postprocess import predict_boxes, simple_map50
from analysis_utils import count_params, model_size_mb, measure_latency

with open("../checkpoints/run_config.json") as f:
    cfg = json.load(f)
CLASS_NAMES, IMG_SIZE, DATA_ROOT, BATCH_SIZE = cfg["class_names"], cfg["img_size"], cfg["data_root"], cfg["batch_size"]
NUM_CLASSES = len(CLASS_NAMES)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open("../checkpoints/compression_log.json") as f:
    compression_log = json.load(f)

## Load both models

The compressed model has to be rebuilt with the same layer-by-layer
factorization before `load_state_dict` — the checkpoint stores the factored shapes.

In [ ]:
original = YOLOv3(num_classes=NUM_CLASSES)
state = torch.load("../checkpoints/yolov3_best.pt", map_location="cpu")
original.load_state_dict(state["model_state"])
original.to(DEVICE).eval()

compressed = YOLOv3(num_classes=NUM_CLASSES)
compressed.load_state_dict(state["model_state"])  # start from same trained weights...
for name, entry in compression_log.items():
    if entry.get("skipped_no_benefit"):
        continue
    conv = dict(compressed.named_modules())[name]
    factored = tucker2_factorize_conv(conv, entry["rank_out"], entry["rank_in"])
    set_module_by_name(compressed, name, factored)
compressed.load_state_dict(torch.load("../checkpoints/yolov3_compressed.pt", map_location="cpu"))
compressed.to(DEVICE).eval()
print("both models loaded")

## Parameter count & model size

In [ ]:
p_orig, p_comp = count_params(original), count_params(compressed)
s_orig, s_comp = model_size_mb(original), model_size_mb(compressed)

print(f"{'':20s}{'params':>15s}{'size (MB)':>15s}")
print(f"{'original':20s}{p_orig:15,d}{s_orig:15.1f}")
print(f"{'compressed':20s}{p_comp:15,d}{s_comp:15.1f}")
print(f"{'reduction':20s}{(1-p_comp/p_orig)*100:14.1f}%{(1-s_comp/s_orig)*100:14.1f}%")

## Inference latency

In [ ]:
lat_orig = measure_latency(original, (1,3,IMG_SIZE,IMG_SIZE), DEVICE)
lat_comp = measure_latency(compressed, (1,3,IMG_SIZE,IMG_SIZE), DEVICE)
print(f"original:   {lat_orig:.2f} ms/image")
print(f"compressed: {lat_comp:.2f} ms/image  ({(1-lat_comp/lat_orig)*100:+.1f}% vs original)")

## Detection accuracy (mAP@0.5)

Note: Tucker-2's 1x1/kxk/1x1 factorization adds sequential ops, so wall-clock
latency doesn't always drop proportionally to parameter count — especially
on GPUs where the original wide convs are already well-parallelized. Param
count / model-size reduction and accuracy retention are usually the more
meaningful numbers for an on-device deployment story; call this out live if
latency doesn't improve as much as the parameter count does.

In [ ]:
val_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                            images_per_class=30, augment=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=yolo_collate_fn, num_workers=2)

def evaluate_map(model):
    all_preds, all_targets = [], []
    for imgs, targets in val_loader:
        imgs = imgs.to(DEVICE)
        preds = predict_boxes(model, imgs, NUM_CLASSES, conf_thresh=0.25)
        all_preds.extend([(p[0].cpu(), p[1].cpu(), p[2].cpu()) for p in preds])
        for t in targets:
            if t.numel() == 0:
                all_targets.append((torch.zeros(0,4), torch.zeros(0, dtype=torch.long)))
                continue
            cx, cy, w, h = t[:,1]*IMG_SIZE, t[:,2]*IMG_SIZE, t[:,3]*IMG_SIZE, t[:,4]*IMG_SIZE
            boxes = torch.stack([cx-w/2, cy-h/2, cx+w/2, cy+h/2], dim=1)
            all_targets.append((boxes, t[:,0].long()))
    return simple_map50(all_preds, all_targets, NUM_CLASSES)

map_orig = evaluate_map(original)
map_comp = evaluate_map(compressed)
print(f"original mAP@0.5:   {map_orig:.3f}")
print(f"compressed mAP@0.5: {map_comp:.3f}  ({(map_comp-map_orig):+.3f})")

## Per-layer compression ratio (which layers got compressed the hardest)

In [ ]:
names = [n for n in compression_log if not compression_log[n].get("skipped_no_benefit")]
ratios = [compression_log[n]["chosen_ratio"] for n in names]
order = sorted(range(len(names)), key=lambda i: ratios[i])[:25]

plt.figure(figsize=(9, 7))
plt.barh([names[i] for i in order][::-1], [ratios[i] for i in order][::-1], color="steelblue")
plt.xlabel("kept ratio (lower = compressed harder)")
plt.title("25 most-compressed layers")
plt.tight_layout()
plt.show()

## Summary table

In [ ]:
import pandas as pd
summary = pd.DataFrame({
    "metric": ["Total params", "Model size (MB)", "Latency (ms/img)", "mAP@0.5"],
    "original": [f"{p_orig:,}", f"{s_orig:.1f}", f"{lat_orig:.2f}", f"{map_orig:.3f}"],
    "compressed": [f"{p_comp:,}", f"{s_comp:.1f}", f"{lat_comp:.2f}", f"{map_comp:.3f}"],
    "change": [f"{(1-p_comp/p_orig)*100:.1f}%", f"{(1-s_comp/s_orig)*100:.1f}%",
               f"{(1-lat_comp/lat_orig)*100:+.1f}%", f"{(map_comp-map_orig):+.3f}"],
})
summary